In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import yfinance as yf

# Fetch real-time market data
def get_stock_data(tickers, start="2020-01-01"):
    data = yf.download(tickers, start=start)
    if 'Adj Close' in data.columns:
        data = data['Adj Close']
    else:
        data = data['Close']
    returns = data.pct_change().dropna()
    return returns

# Define CVaR Calculation (Expected Shortfall at 95% Confidence Level)
def calculate_cvar(returns, weights, alpha=0.05):
    portfolio_returns = returns @ weights
    var_threshold = np.percentile(portfolio_returns, 100 * alpha)
    cvar = portfolio_returns[portfolio_returns <= var_threshold].mean()
    return -cvar  # Negative because we want to minimize it

# Objective Function: Maximize Return while Penalizing CVaR
def objective_function(weights, returns, lambda_cvar):
    portfolio_return = np.dot(weights, returns.mean())
    cvar = calculate_cvar(returns, weights)
    return -portfolio_return + lambda_cvar * cvar  # Tradeoff between return & CVaR

# Fetch data and compute returns
tickers = ["TSLA", "BND", "SPY"]
returns = get_stock_data(tickers)

# Constraints & Bounds
num_assets = len(tickers)
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})  # Weights sum to 1
bounds = [(0, 1) for _ in range(num_assets)]  # No short selling

# Initial Weights
initial_weights = np.array([1/num_assets] * num_assets)

# Perform Optimization
lambda_cvar = 10  # Adjust to balance return vs. risk
optimized = minimize(objective_function, initial_weights, args=(returns, lambda_cvar), 
                     method='SLSQP', bounds=bounds, constraints=constraints)

# Extract Optimal Weights & Metrics
optimal_weights = optimized.x
optimal_return = np.dot(optimal_weights, returns.mean())
optimal_cvar = calculate_cvar(returns, optimal_weights)

# Display Results
print("Optimized Portfolio Weights:", optimal_weights)
print(f"Expected Return: {optimal_return:.4%}")
print(f"Conditional Value at Risk (CVaR 95%): {optimal_cvar:.4%}")

# Maximum Drawdown Calculation
def calculate_maximum_drawdown(returns):
    cumulative_returns = (1 + returns).cumprod()
    peak = cumulative_returns.cummax()
    drawdown = (cumulative_returns - peak) / peak
    max_drawdown = drawdown.min()
    return max_drawdown

max_drawdown = calculate_maximum_drawdown(returns @ optimal_weights)
print(f"Maximum Drawdown (MDD): {max_drawdown:.4%}")

# Stress Testing (Simulating Extreme Market Events)
def stress_test(returns, weights, shock=-0.3):
    stressed_returns = returns * (1 + shock)  # Apply a market shock
    stressed_cvar = calculate_cvar(stressed_returns, weights)
    return stressed_cvar

stress_cvar = stress_test(returns, optimal_weights)
print(f"Stressed CVaR (Market Shock -30%): {stress_cvar:.4%}")

# Beta & Correlation Analysis
def beta_correlation(returns, weights, market_returns):
    portfolio_returns = returns @ weights
    beta = np.cov(portfolio_returns, market_returns)[0, 1] / np.var(market_returns)
    correlation = np.corrcoef(portfolio_returns, market_returns)[0, 1]
    return beta, correlation

market_returns = returns.mean(axis=1)  # Placeholder, replace with actual market returns
beta, correlation = beta_correlation(returns, optimal_weights, market_returns)
print(f"Portfolio Beta: {beta:.4f}")
print(f"Portfolio-Market Correlation: {correlation:.4f}")


[*********************100%***********************]  3 of 3 completed


Optimized Portfolio Weights: [0.95861004 0.04138996 0.        ]
Expected Return: 0.0037%
Conditional Value at Risk (CVaR 95%): 0.9595%
Maximum Drawdown (MDD): -17.7356%
Stressed CVaR (Market Shock -30%): 0.6717%
Portfolio Beta: 0.0785
Portfolio-Market Correlation: 0.3064
